# 07. Difficulty Routing — Cutting Frontier Token Cost

Frontier models like Sonnet are powerful but expensive ($3/$15 per MTok). Yet a large share of customer inquiries are **questions a small model answers perfectly well** — business hours, how to report a lost card. In this notebook the decision layer becomes a **traffic router**: we classify each question's difficulty with `/v1/systemone` and send **simple ones to Haiku 4.5 ($1/$5), complex ones to Sonnet 4.5 ($3/$15)**. Generation stays on Bedrock in both arms — only the router is a self-hosted decision model.

Relation to notebook 05 (Bedrock Intelligent Prompt Routing):

| | Bedrock IPR (notebook 05) | difficulty routing (this notebook) |
|---|---|---|
| picks between | Bedrock models of one family (Lite/Pro) | **two models of different cost tiers (Haiku vs Sonnet)** — no family constraint |
| router | managed (built-in predictor) | built by you (decision layer) |
| router cost | $1 / 1,000 requests | tens to hundreds of ms, ~zero marginal cost |
| control | one responseQualityDifference knob | you design the rubric, choices, fallback |

The two are complementary — use IPR when managed is enough; use the decision layer when you want to own the rubric and the fallback direction.


In [1]:
import os
os.environ["DMWS_LANG"] = "en"   # localize helper output/graph captions (set BEFORE importing ws_common)
import sys, urllib.request; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common path (relative inside the repo)
import ws_common as W
print("decision server:", f"http://{W.STRANDS_HOST}:{W.STRANDS_PORT}")
print("health:", urllib.request.urlopen(f"http://{W.STRANDS_HOST}:{W.STRANDS_PORT}/health", timeout=5).status)
client, sonnet = W.make_clients()
haiku  = W.BedrockLLM(client, W.BEDROCK_MODEL_IDS["haiku-4-5"], W.PRICING["haiku-4-5"])
router = W.SystemOneDecision()

decision server: http://localhost:8004
health: 200


## 1. The router = a 2-choice decision

We call `/v1/systemone` with a difficulty rubric (`W.ROUTER_QUESTION`) and the choices `["complex", "simple"]`.

> **Fallback guardrail**: the order of the choices matters. If the router output is invalid, the decision backend falls back to `choices[0]` — and `complex` is first, so **errors always route to Sonnet (the high-quality side)**. Cost optimization must never become a quality incident.


In [2]:
print("rubric:", W.ROUTER_QUESTION, "\n")
for q in (W.routing_questions()[0]["q"], W.routing_questions()[5]["q"]):
    d = router.decide(state_text=q, question=W.ROUTER_QUESTION,
                      choices=W.ROUTER_CHOICES, decision_id="route")
    print(f"[{d.label:<7s}] conf {d.confidence:.3f} {d.latency_ms:>6,.0f}ms | {q[:60]}")

rubric: Classify the difficulty of this banking customer inquiry. simple = a single fact or procedure that can be answered in one or two sentences (business hours, where a menu is, how to report/change one thing). complex = requires multi-step reasoning, comparing options, interpreting policy/tax/fee rules, or judgment about the customer's specific situation. 

[simple ] conf 0.960    156ms | What are your customer-service hours?


[complex] conf 0.726    110ms | I am moving my salary account from Bank A to Bank B and need


## 2. The question set — 10 mixed-difficulty questions

5 simple FAQ questions and 5 that need multi-step reasoning or policy interpretation. `gold` is the intended label.


In [3]:
from IPython.display import Markdown, display
rows = ["| # | question | gold |", "|---|---|---|"]
for i, item in enumerate(W.routing_questions(), 1):
    rows.append(f"| {i} | {item['q']} | {item['gold']} |")
display(Markdown("\n".join(rows)))

| # | question | gold |
|---|---|---|
| 1 | What are your customer-service hours? | simple |
| 2 | Where in the app can I change my debit-card PIN? | simple |
| 3 | I lost my card. How do I report it lost? | simple |
| 4 | Can I withdraw cash from ATMs abroad? | simple |
| 5 | Where can I get a balance certificate issued? | simple |
| 6 | I am moving my salary account from Bank A to Bank B and need to move 7 autopay registrations and my card billing account with it. What order should I do this in to avoid missed payments or penalties? | complex |
| 7 | My overseas transfer arrived short because of intermediary bank fees. Explain the difference between OUR/SHA/BEN fee options and whether the deducted amount can be refunded. | complex |
| 8 | For exchanging USD 5,000, which is better: an 80% FX-rate discount or a transfer-fee waiver? Compare them with the calculation. | complex |
| 9 | I am a sole proprietor with USD 250k annual revenue. How do debt-service-ratio rules apply when I hold both a business loan and a personal credit loan? | complex |
| 10 | I want to transfer USD 400 every month into my minor child's account. What is the gift-tax reporting threshold and the filing procedure? | complex |

## 3. Baseline — all 10 questions to Sonnet 4.5

First, measure the token cost with no routing: everything goes to the frontier model. `BedrockLLM` computes `cost_usd` from the actual token counts of each call.


In [4]:
baseline = []
for item in W.routing_questions():
    r = sonnet.generate(system=W.GEN_SYSTEM, user=item["q"], max_tokens=300)
    baseline.append({"backend": "sonnet", "cost_usd": r.cost_usd,
                     "input_tokens": r.usage["input_tokens"],
                     "output_tokens": r.usage["output_tokens"],
                     "latency_ms": r.latency_ms})
    print(f"${r.cost_usd:.5f} {r.latency_ms:>7,.0f}ms | {item['q'][:55]}")
print(f'\ntotal: ${sum(b["cost_usd"] for b in baseline):.4f}')

$0.00217   3,095ms | What are your customer-service hours?


$0.00292   3,245ms | Where in the app can I change my debit-card PIN?


$0.00256   3,030ms | I lost my card. How do I report it lost?


$0.00346   4,153ms | Can I withdraw cash from ATMs abroad?


$0.00241   2,771ms | Where can I get a balance certificate issued?


$0.00477   5,627ms | I am moving my salary account from Bank A to Bank B and


$0.00475   5,989ms | My overseas transfer arrived short because of intermedi


$0.00474   4,081ms | For exchanging USD 5,000, which is better: an 80% FX-ra


$0.00474   5,156ms | I am a sole proprietor with USD 250k annual revenue. Ho


$0.00473   5,425ms | I want to transfer USD 400 every month into my minor ch

total: $0.0373


## 4. Routed — simple to Haiku, complex to Sonnet

The same 10 questions, now through the router. Haiku calls are billed Bedrock tokens too, so we record the **real token cost of every call** — the savings come entirely from "did a cheaper model answer the same question".


In [5]:
routed, correct = [], 0
for item in W.routing_questions():
    d = router.decide(state_text=item["q"], question=W.ROUTER_QUESTION,
                      choices=W.ROUTER_CHOICES, decision_id="route")
    correct += (d.label == item["gold"])
    target, name = (haiku, "haiku") if d.label == "simple" else (sonnet, "sonnet")
    r = target.generate(system=W.GEN_SYSTEM, user=item["q"], max_tokens=300)
    routed.append({"backend": name, "cost_usd": r.cost_usd,
                   "input_tokens": r.usage["input_tokens"],
                   "output_tokens": r.usage["output_tokens"],
                   "latency_ms": r.latency_ms, "router_ms": d.latency_ms})
    print(f"route={d.label:<7s} (conf {d.confidence:.2f}, {d.latency_ms:>5,.0f}ms)"
          f" -> {name:<6s} ${r.cost_usd:.5f} | {item['q'][:45]}")
print(f'\nrouter accuracy vs gold: {correct}/10')

route=simple  (conf 0.96,   113ms) -> haiku  $0.00065 | What are your customer-service hours?


route=simple  (conf 0.92,   114ms) -> haiku  $0.00109 | Where in the app can I change my debit-card P


route=simple  (conf 0.92,   114ms) -> haiku  $0.00120 | I lost my card. How do I report it lost?


route=simple  (conf 0.65,   112ms) -> haiku  $0.00145 | Can I withdraw cash from ATMs abroad?


route=simple  (conf 0.84,   915ms) -> haiku  $0.00124 | Where can I get a balance certificate issued?


route=complex (conf 0.73,   113ms) -> sonnet $0.00477 | I am moving my salary account from Bank A to 


route=complex (conf 0.82,   113ms) -> sonnet $0.00475 | My overseas transfer arrived short because of


route=complex (conf 0.83,   122ms) -> sonnet $0.00474 | For exchanging USD 5,000, which is better: an


route=complex (conf 0.93,   113ms) -> sonnet $0.00474 | I am a sole proprietor with USD 250k annual r


route=complex (conf 0.70,   113ms) -> sonnet $0.00460 | I want to transfer USD 400 every month into m

router accuracy vs gold: 10/10


## 5. Cost comparison — honest accounting

We look at the token-cost savings and the monthly projection. The routing decisions themselves run on the GPU the decision layer already uses, so **the router adds zero cost here** — if you were to provision a GPU for routing alone, you would weigh that fixed cost against the savings (see the note under the table).


In [6]:
report = W.routing_cost_report(baseline, routed, req_per_day=10_000)

| metric | all-Sonnet baseline | difficulty-routed |
|---|---:|---:|
| questions | 10 | 10 |
| Sonnet / Haiku calls | 10 / 0 | 5 / 5 |
| Bedrock tokens (in/out) | 687/2,346 | 687/2,563 |
| Bedrock token cost (10 questions) | $0.0373 | $0.0292 |
| monthly token cost (10,000 req/day) | $1,118 | $877 |

**Token-cost savings: 21.5%** ($240/month at 10,000 req/day)

Honest accounting: the routing decisions run on the decision-layer GPU (ml.g5.2xlarge, ~$1.515/h, ~$1,091/month) that this architecture already provisions for the agent's D1-D8 decisions, so the router itself adds no per-call Bedrock cost. If you provisioned the GPU for routing alone, weigh that fixed cost against the monthly savings above.

### What to look at

- **Quality guardrail**: every complex question was answered by Sonnet, and the fallback on router error is `choices[0]=complex` (Sonnet) too. Savings come only from questions a small model handles well.
- **The router itself is nearly free**: tens to hundreds of ms per decision, no Bedrock call. Compare with IPR's $1 per 1,000 requests.
- **The price ratio caps the savings**: Haiku costs 1/3 of Sonnet, so with a 50% simple share the arithmetic ceiling is ~33%. The real number depends on the output-token distribution — compare with the measured value above.
- **It is sensitive to rubric wording**: the same lesson as notebook 05 — changing the simple/complex criteria changes the routing distribution. Validate accuracy on a sample of real traffic before production.
- The savings scale with the **share of simple traffic**. This lab uses a 5:5 mix; recompute with your real FAQ share (usually higher).


### Try it yourself

- Change `req_per_day` and watch the monthly savings move.
- 3-way routing: extend the choices to `["complex", "medium", "simple"]` and answer simple ones with a **canned FAQ reply and no model call at all** — how far does the savings rate go?
- Confidence-based routing: build a conservative policy that sends low-confidence `simple` verdicts to Sonnet anyway (possible precisely because the confidence is calibrated).
- Feed scenario utterances like `W.scenario(1)["question"]` to the router and see how they classify.
